# Exploration of Biobank lookups data

In [ ]:
import pandas as pd
import json
import re

In [ ]:
import numpy as np
from datetime import datetime

def display_special_chars_stats(df, column, additional_columns = [], not_num_sample_sz = 5, num_sample_sz = 3, extra_patterns = None):
    df = df[[column] + additional_columns].copy()
    special_chars = set()
    df['special_chars'] = df[column].apply(lambda s: re.findall(r'[^A-Za-z0-9 ]', str(s)))
    for _, val in df['special_chars'].items():
        special_chars.update(val)
    print('Found special chars:')
    print(' '.join(sorted(special_chars)))
    print(f'count: {len(special_chars)}')

    rand_seed = int(datetime.now().timestamp())

    df['rand'] = np.random.RandomState(seed=rand_seed).uniform(size=len(df))
    df = df.sort_values('rand')

    rows = []

    for char in special_chars:
        # matched = df[df['special_chars'].apply(lambda lst: char in lst)]
        # matched = matched.head(sample_sz).copy()
        
        regex = re.compile(rf'([^0-9]|^){re.escape(char)}([^0-9]|$)')
        not_num_matched = df[df[column].str.contains(regex, na=False)]
        not_num_matched = not_num_matched.head(not_num_sample_sz).copy()

        regex = re.compile(rf'([0-9]{re.escape(char)}|{re.escape(char)}[0-9])')
        # regex = re.compile(rf'([0-9]{re.escape(char)}[0-9])')
        num_matched = df[df[column].str.contains(regex, na=False)]
        num_matched = num_matched.head(num_sample_sz).copy()

        matched = pd.concat([not_num_matched, num_matched], ignore_index=True)
        matched['spec_char'] = char        
        rows.append(matched)
    
    if extra_patterns is None:
        extra_patterns = []
    if not isinstance(extra_patterns, list):
        extra_patterns = [extra_patterns]
    for pattern in extra_patterns:
        regex = re.compile(pattern)
        matched = df[df[column].str.contains(regex, na=False)]
        matched = matched.head(not_num_sample_sz).copy()
        matched['spec_char'] = pattern
        rows.append(matched)

    joined = pd.concat(rows, ignore_index=True)
    joined = joined.drop(columns=['rand', 'special_chars']).sort_values('spec_char')

    with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150):
        display(joined)

In [ ]:
extra_patterns = ['[0-9],[0-9]{3}', '[0-9],[0-9]{1,2}[^0-9]', '[^A-Za-z0-9]\.[0-9]+', '[0-9]/[0-9]']

## BNF Lookup `bnf_lkp.csv`

In [ ]:
bnf_df = pd.read_csv('../../data/lkps/bnf_lkp.csv', sep=',', dtype=str)
bnf_cols = ['BNF_Presentation_Code', 'BNF_Presentation', 'BNF_Product', 'BNF_Chemical_Substance']

### Special characters in BNF_Chemical_Substance column

In [ ]:
display_special_chars_stats(bnf_df, 'BNF_Chemical_Substance', [], 5, 3, extra_patterns)

### Special characters in BNF_Product column

In [ ]:
display_special_chars_stats(bnf_df, 'BNF_Product', [], 5, 3, extra_patterns)

### Special characters in BNF_Presentation column

In [ ]:
display_special_chars_stats(bnf_df, 'BNF_Presentation', [], 5, 3, extra_patterns)

### BNF Conslusions

Special characters identified in given column:

- BNF_Chemical_Substance: `& ' ( ) + , - . /`
- BNF_Product: `% & ' ( ) + , - . / : _`
- BNF_Presentation: `# % & ' ( ) * + , - . / : @ _`

Special care should be taken with following values.

**BNF_Chemical_Substance:**

- `HumPapvirus(T-6,11,16,18,31,33,45,52,58)`
- `B.J.6.`
- `Carbomer 940/980`

**BNF_Product**:

- `Aidulan (30mcg/75mcg)`
- `HPV (Type 6,11,16,18,31,33,45,52,58)`
- `M.C.V. 9+3`
- `Baclospas-10`
- `Bm-Test 1-44 (Reagent)`
- `Nindaxa 2.5`
- `Nonoxinol 10/Nonoxinol 11`
- `Gedarel (30/150mcg)`
- `Triamaxco 50/25`
- `Neocon 1/35`

**BNF_Presentation:**

- `VSL#3_Probiotic Food Supp Pdr Sach 4.4g`
- `Lamb_Vit C & Bioflav Tab 1.5g/75mg(8137)`
- `Neo-Cytamen_'1000' Inj 1mg/ml 1ml Amp`
- `Coloplast SpeediCath Compact Set Male Size12/18(20-Pack)Cath`
- `Plegridy_Inj63mcg/0.5ml+94mcg/0.5mlPfPen`
- `Natures Nectar_Act-Med 14+ Manuka Honey`
- `Coloplast Self-Cath Plus Fle Size 8-16 (25-Pack) Cath`
- `5.F.U._Inf (Multi-Day)`
- `Payne_Material Flng Support Mk 7,8 85cm`
- `HPV (Type 6,11,16,18)_Vac 0.5ml Pfs`
- `KetoCal 4:1_LQ Liq (Unflav)`
- `Menotrophin_Inj 600u 1:1 Vl + Dil`
- `PropyleneGlycol 40%/Clobetasol .05%_Oint`
- `Baxter_Pot/Sod Chlor/Glucose .5L (B1723)`
- `Benzocaine/Mepyramine Mal_Spy 1/.5% 30ml`
- `Steriflex_No12 Pot/SodChlor.15/.9%Inf 1L`
- `Baxter_Chlorhex/Cetrimide .015%/.15% 1L`
- `Salbut/Ipratrop_NebSoln 2.5/.5mg Ud2.5ml`
- `Benzocaine/Mepyramine Mal_Spy 1/.5% 30ml`
- `Amoxicillin_Tab 500mg @gn`
- `Power_Super Glucosam+Chond Cap 500/400mg`
- `Dansac_NovaLife TRE 1 Clsd Colo Bag Midi ClrC/Fit 40-55/70mm`
- `Fluticasone/Formoterol_Inh 250/10mcg120D`
- `Glyceryl Trinit_Duo/P Spy 400mcg 250/75D`
- `Septanest_Inj 1:200,000/2.2ml Cart`
- `Sterculia/Frangula_Gran 62/8%Sach 7g G/F`
- `Hypromellose/Dextran 70_Eye Dps.3/.1% Ud`

## Read v2 Lookup `read_v2_drugs_lkp.csv`

In [ ]:
read2_df = pd.read_csv('../../data/lkps/read_v2_drugs_lkp.csv', sep=',', dtype=str)

### Special characters in term_description

In [ ]:
display_special_chars_stats(read2_df, 'term_description', [], 5, 3, extra_patterns)

### Read v2 Conclusions

Special characters identified: `! " # % & ' ( ) * + , - . / : [ ] ^`

Special care should be taken with following values:

- `LICENCE SERIAL NO. !^!^!^!^`
- `AMOXICILLIN [2]`
- `HAINSWORTH 1"/NSI 63 colostomy bags x20`
- `AQUADRY 785539/1.5" standard rubber ileo flanges x1`
- `PHOSPHATES FORMULA "B" enema 128mL`
- `ROSE RUBBER '73' large ileostomy bags x1`
- `NELATON CATHETER STERILE (6)`
- `MEPIVACAINE HYDROCHLORIDE+ADRENALINE 2%/1:100,000 cartridges`
- `ADRENALINE 1:10,000 100micrograms/1mL solution for injection`
- `FIFTY:50 ointment 250g`
- `BUPIVACAINE HYDROCHLORIDE+ADRENALINE 0.5%/5micrograms/mL(1:200,000) injection 10mL`
- `JOB ALLEVIANT size 6/9231 C2(23-32mmHg) sht c/toe thigh gmt`
- `HUMULIN M5 50/50 cartridges 3mL`
- `MEDIVEN PLUS DT117/0/II class 1(18-21mmHg) beige right thigh length...`

## CTV3 Lookup `read_ctv3_lkp.csv`

**Important notice**: We use only medication (drug) part of CTV3, no clinical events.

In [ ]:
read3_df = pd.read_csv('../../data/lkps/read_ctv3_lkp.csv', sep=',', dtype=str)
read3_df = read3_df.dropna(subset=['read_code'])
read3_df = read3_df[read3_df['read_code'].str.match(r'^[a-z]')]

### Special characters in term_description

In [ ]:
display_special_chars_stats(read3_df, 'term_description', ['read_code'], 5, 3, extra_patterns)

### CTV3 Conclusions

**Important notice**: We use only medication (drug) part of CTV3, no clinical events.

Special characters identified only in medication/drug section:`! " # % & ' ( ) + , - . / : > [ ] ^`

Special characters identified in whole CTV3 lookup:`! " # % & ' ( ) * + , - . / : ; < = > ? [ ] ^`

Special care should be taken with following values (only medication/drug section):
- `Licence serial no. !^!^!^!^`
- `Salt PP ZL0053 1"/0.75" spare flange+sheath`
- `Hollister 1.5"/7443 urostomy bags x20`
- `VSL#3 probiotic powder sachets`
- `Triamcinolone acetonide+nystatin 0.1%/100000units/g cream`
- `Ellis, Son & Paramore Ltd`
- `Bard male 14ounce/0005 urinal for day & night use x1`
- `Nonoxinol '9' 12.5% foam`
- `VENOTRAIN MICRO 21880011220234 C1(18-21mmHg) o/toe b/knee`
- `Bupivacaine hydrochloride 50mg/10mL(0.5%) injection`
- `LYXUMIA 10mcg/0.2mL+20mcg/0.2mL soln inj prefilled pens 3mL`
- `STAR COTTON VII/363-1058 C1(18-21mmHg) thigh lymph garment`
- `Femoston 2/20 tablet`
- `Bupivacaine hydrochloride+adrenaline 0.5%/50micrograms/10mL(1:200000) injection`
- `Iodinated[125I] human albumin >148kBq/mL injection+syringe`

## DM+D Lookup `dmd_lkp.csv`

In [ ]:
dmd_df = pd.read_csv('../../data/lkps/dmd_lkp.csv', sep=',', dtype=str)

### Special characters in term

In [ ]:
display_special_chars_stats(dmd_df, 'term', [], 5, 3, extra_patterns)

### DM+D Conclusions

Special characters identified: `! " # % & ' ( ) + , - . / : < > [ ] ^`

Special care should be taken with following values:

- `Licence serial no. !^!^!^!^`
- `Bupivacaine 100mg/20ml / Adrenaline 100micrograms/20ml (1 in 200,00) solution for injection ampoules 1 ampoule (prod...`
- `Bupivacaine 100mg/20ml / Adrenaline 100micrograms/20ml (1 in 200,00) solution for injection ampoules (product)`
- `Bullen UF52 1.5"/medium belt+aluminium retaining shield`
- `Aquadry 785539/1.5" standard ileo flanges x1`
- `Phosphates formula "b" enema 128mL`
- `%v/v (qualifier value)`
- `Dermablend cover creme chroma 4 1/2 golden bronze (Brodie & Stone Plc) (product)`
- `Hypromellose 0.3% / Dextran '70' 0.1% eye drops 0.4ml unit dose preservative free (product)`
- `Memantine 5mg/10mg/15mg/20mg tablets treatment initiation pack (A A H Pharmaceuticals Ltd) 28 tablet 1 x (7tabs+7tabs+7tabs+7tabs) (product)`
- `KetoCal 4:1LQ liquid vanilla (Nutricia Ltd) 237 ml (product)`
- `Natrelle 410 cohesive N-27-FX145-615 silicone gel-filled textured anatomical full height extra-full projection breast implant 615g (Allergan)<New ...`
- `Iodinated[125I] human albumin >148kBq/mL injection+syringe - product`
- `Impleo IMP-EHR 240 cohesive silicone gel-filled textured round extra high profile breast implant 235g (Nagor)> (physical object)`
- `Histoacryl skin adhesive (B.Braun Medical Ltd) .5 ml (product)`
- `3,4-Diaminopyridine 20mg tablets 1 tablet (product)`
- `Premier Omega 3,6,9 1000mg capsules (Premier Health Products Ltd) (product)`